In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn import preprocessing
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF
from sklearn.decomposition import PCA
from pathlib import Path

In [ ]:
spectrum_files = sorted(
    Path("data/train").glob("cl_*.txt"),
    key=lambda path: int(path.stem.split("_")[1]),
)

# Column 0 = TT, column 1 = TE, column 2 = EE
tt_spectra = np.stack([
    np.loadtxt(path)[:, 0]
    for path in spectrum_files
])

cosmologies = np.loadtxt("data/train/cosmologies.txt")

print(tt_spectra.shape)
# (number of cosmologies, number of multipoles)

In [ ]:
new_cosmo = np.delete(cosmologies, 3, axis = 1) #removing As from the cosmologies

rng = np.random.default_rng(42)
all_indices = rng.permutation(len(new_cosmo))

n_gp_train = 1000
n_validation = 1000

train_indices = all_indices[:n_gp_train]
validation_indices = all_indices[
    n_gp_train:n_gp_train + n_validation
]

print("training:", len(train_indices))
print("validation:", len(validation_indices))
#splitting indices into subset and training for internal
# validation

assert len(np.intersect1d(
    train_indices,
    validation_indices,
)) == 0

In [ ]:
#Standardizing the cosmologies
scaler = preprocessing.StandardScaler()

X_train_scaled = scaler.fit_transform(new_cosmo[train_indices])
X_validation_scaled = scaler.transform(new_cosmo[validation_indices])
#transforming both, but only fitting with the training indices

In [ ]:
As = 1e-10 * np.exp(cosmologies[:,3])


tt_lognorm = np.log(
    np.clip(tt_spectra / As[:,None],1e-3, None)
) # log(Cl / As)

In [ ]:
pca = PCA(n_components=30) #dimension 30 outputs, 10 is too small


#for internal validation, only fit PCA with subset
tt_coefficients = pca.fit_transform(tt_lognorm[train_indices]) #3000 to 10 dimensions
# transforming the validation coefficients, not fitting though
tt_coefficients_validation = pca.transform(tt_lognorm[validation_indices])
print(tt_coefficients.shape)
print(tt_coefficients_validation.shape)

In [ ]:
def train_gp_models(X_train, coefficients):

    models = []

    # goes through each of the PCA coefficients and has a separate 6 length scales per coefficient.
    for component in range(coefficients.shape[1]):
        kernel = ConstantKernel(1.0) * RBF(
            length_scale=np.ones(X_train.shape[1]), #5 separate length scales for 1 coefficient
            length_scale_bounds = (1e-2, 1e2),
        )
        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha = 1e-8,
            normalize_y = True,
            n_restarts_optimizer=2,
            random_state = 42,
        )

        gp.fit(X_train, coefficients[:,component]) #fits the training cosmologies (X_train) and PCA coefficients
        models.append(gp)

    return models

In [ ]:
tt_models = train_gp_models(
    X_train_scaled,
    tt_coefficients,
)#truncated spectra, to see if it works

print(len(tt_models))

In [ ]:
def predict_models(models, X):
    predictions = [
        gp.predict(X) 
        for gp in models
    ] #actually predict the coefficients

    return np.column_stack(predictions)

In [ ]:

# trying to predict the tt_coefficients for the validation cosmologies
tt_coefficients_predicted = predict_models(
    tt_models,
    X_validation_scaled,
)

In [ ]:
# now need to reverse the PCA to log(C_ell / As), then undo log and As division
tt_lognorm_pred = pca.inverse_transform(tt_coefficients_predicted)
As_valid = As[validation_indices]

tt_predicted = np.exp(
    tt_lognorm_pred) * As_valid[:, None]

tt_validation = tt_spectra[validation_indices]

In [ ]:
relative_error = np.abs(
    tt_predicted - tt_validation
) / np.maximum(np.abs(tt_validation),1e-12)

print("Median relative error:", np.median(relative_error))
print(
    "95th percentile relative error:",
    np.percentile(relative_error, 95),
)

print(
    "Fraction below 1%:",
    np.mean(relative_error < 0.01),
)

In [ ]:
# Best reconstruction possible with the fitted 10-component PCA
tt_pca_only_lognorm = pca.inverse_transform(
    tt_coefficients_validation
)

tt_pca_only = (
    np.exp(tt_pca_only_lognorm)
    * As_valid[:, None]
)

pca_only_error = np.abs(
    tt_pca_only - tt_validation
) / np.maximum(np.abs(tt_validation), 1e-12)

print(
    "PCA-only median error:",
    np.median(pca_only_error),
)

In [ ]:
for component, model in enumerate(tt_models):
    print(
        f"PC {component + 1}:",
        model.kernel_,
    )